# Encrypted Search Researcher Tutorial

This notebook demonstrates a real developer workflow for encrypted-search RAG experiments in SecureRAG.

You will:
- Build encrypted corpora using built-in schemes (`sse`, `structured`).
- Inspect encrypted query artifacts and validate plaintext does not leak in tokens.
- Prototype and register a custom encrypted scheme in-notebook.
- Run side-by-side retrieval comparisons across schemes.
- Validate version-guard behavior for legacy encrypted indexes.

In [5]:
from __future__ import annotations

from pathlib import Path
import hashlib
import hmac
import re
import socket
import subprocess
import sys
import time

import httpx

# Make the local workspace importable when running this notebook directly.
cwd = Path.cwd().resolve()
repo_root = next(
    (p for p in [cwd, *cwd.parents] if (p / "pyproject.toml").exists() and (p / "securerag").is_dir()),
    None,
 )
if repo_root is None:
    raise RuntimeError("Could not locate repository root containing pyproject.toml and securerag/")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from securerag import PrivacyConfig, PrivacyProtocol, SecureRAGAgent
from securerag.backend_client import create_backend
from securerag.corpus import CorpusBuilder
from securerag.errors import BackendError
from securerag.llm import OllamaLLM
from securerag.models import RawDocument
from securerag.scheme_plugin import EncryptedSchemePlugin


def free_port() -> int:
    with socket.socket() as s:
        s.bind(("127.0.0.1", 0))
        return int(s.getsockname()[1])


def launch_sim_server(port: int) -> subprocess.Popen:
    return subprocess.Popen(
        [
            sys.executable,
            "-m",
            "uvicorn",
            "securerag.sim_server:app",
            "--host",
            "127.0.0.1",
            "--port",
            str(port),
        ],
        cwd=str(repo_root),
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
    )


def wait_for_health(base_url: str, proc: subprocess.Popen | None = None, timeout: float = 10.0) -> None:
    deadline = time.time() + timeout
    while time.time() < deadline:
        if proc is not None and proc.poll() is not None:
            stderr_out = ""
            if proc.stderr is not None:
                stderr_out = proc.stderr.read().strip()
            raise RuntimeError(
                f"sim_server exited early with code {proc.returncode}. stderr:\n{stderr_out}"
            )
        try:
            r = httpx.get(f"{base_url}/health", timeout=0.4)
            if r.status_code == 200:
                return
        except Exception:
            pass
        time.sleep(0.1)
    raise RuntimeError(f"sim_server did not become healthy at {base_url}")

In [6]:
port = free_port()
base_url = f"http://127.0.0.1:{port}"
sim_proc = launch_sim_server(port)
wait_for_health(base_url, proc=sim_proc)

docs = [
    RawDocument(doc_id="q3", text="Q3 risk report highlights vendor concentration and delayed remediation."),
    RawDocument(doc_id="policy", text="Security policy mandates quarterly risk treatment and owner assignment."),
    RawDocument(doc_id="ops", text="Incident patterns suggest ingestion queue saturation under peak traffic."),
]

print("sim_server ready at", base_url)

sim_server ready at http://127.0.0.1:62598


## 1) Build and Query with SSE Scheme

In [7]:
sse_corpus = (
    CorpusBuilder(PrivacyProtocol.ENCRYPTED_SEARCH, backend_url=base_url)
    .with_encrypted_search_scheme("sse")
    .with_chunk_size(180)
    .add_documents(docs)
    .build_local(workers=2)
)

sse_cfg = PrivacyConfig(
    protocol=PrivacyProtocol.ENCRYPTED_SEARCH,
    backend=base_url,
    top_k=3,
    encrypted_search_scheme="sse",
)

sse_agent = SecureRAGAgent.from_config(
    sse_cfg,
    corpus=sse_corpus,
    llm=OllamaLLM(use_ollama=False),
)

sse_docs = sse_agent.retriever.retrieve("q3 risk vendor concentration", round_n=0)
print([d.doc_id for d in sse_docs[:3]])
print(sse_docs[0].text[:140])

['q3', 'policy']
Q3 risk report highlights vendor concentration and delayed remediation.


In [8]:
plugin = sse_corpus.extras["plugin"]
key = sse_corpus.extras["enc_key"]
encrypted_query = plugin.encrypt_query("q3 risk vendor concentration", key)

print("Encrypted query fields:", list(encrypted_query.keys()))
print("Encrypted term sample:", encrypted_query["enc_terms"][0][:20] + "...")
print("Any plaintext token leaked?", any(tok in str(encrypted_query) for tok in ["q3", "risk", "vendor"]))

Encrypted query fields: ['enc_terms']
Encrypted term sample: 4ec8a79a0da41174ff39...
Any plaintext token leaked? False


## 2) Structured Scheme with Bigram Toggle

Structured retrieval can be explored with and without bigrams for ablation-style studies.

In [9]:
structured_on = (
    CorpusBuilder(PrivacyProtocol.ENCRYPTED_SEARCH, backend_url=base_url)
    .with_encrypted_search_scheme("structured", structured_use_bigrams=True)
    .add_documents(docs)
    .build_local(workers=2)
)

structured_off = (
    CorpusBuilder(PrivacyProtocol.ENCRYPTED_SEARCH, backend_url=base_url)
    .with_encrypted_search_scheme("structured", structured_use_bigrams=False)
    .add_documents(docs)
    .build_local(workers=2)
)

print("bigrams ON:", structured_on.extras["plugin"].use_bigrams)
print("bigrams OFF:", structured_off.extras["plugin"].use_bigrams)

bigrams ON: True
bigrams OFF: False


In [10]:
class PrefixHMACScheme(EncryptedSchemePlugin):
    """Minimal custom encrypted scheme for prototyping.

    Uses HMAC terms with a fixed prefix to keep token namespace explicit.
    """

    def generate_key(self) -> str:
        return "00112233445566778899aabbccddeeff"

    def _tokenize(self, text: str) -> list[str]:
        return re.findall(r"[a-z0-9]+", text.lower())

    def _enc(self, token: str, key: str) -> str:
        digest = hmac.new(key.encode("utf-8"), token.encode("utf-8"), hashlib.sha256).hexdigest()
        return "pfx:" + digest

    def prepare_chunk(self, text: str, key: str) -> dict:
        terms = [self._enc(t, key) for t in self._tokenize(text)]
        return {"enc_terms": sorted(set(terms))}

    def build_server_index(self, chunks: list[dict]) -> dict:
        rows = []
        inv: dict[str, list[int]] = {}
        for i, c in enumerate(chunks):
            terms = c.get("scheme_data", {}).get("enc_terms", [])
            rows.append({
                "doc_id": c["doc_id"],
                "text": c["text"],
                "metadata": c.get("metadata", {}),
                "terms": terms,
            })
            for t in terms:
                inv.setdefault(t, []).append(i)
        return {"rows": rows, "inv": inv}

    def encrypt_query(self, query: str, key: str) -> dict:
        return {"enc_terms": [self._enc(t, key) for t in self._tokenize(query)]}

    def search(self, server_index: dict, encrypted_query: dict, top_k: int) -> list[dict]:
        q_terms = encrypted_query.get("enc_terms", [])
        counts: dict[int, int] = {}
        for t in q_terms:
            for idx in server_index["inv"].get(t, []):
                counts[idx] = counts.get(idx, 0) + 1

        scored = []
        q_len = max(1, len(q_terms))
        for idx, inter in counts.items():
            row = server_index["rows"][idx]
            union = q_len + len(row["terms"]) - inter
            score = inter / max(1, union)
            scored.append({
                "doc_id": row["doc_id"],
                "text": row["text"],
                "metadata": row["metadata"],
                "score": score,
            })
        scored.sort(key=lambda x: x["score"], reverse=True)
        return scored[:top_k]


EncryptedSchemePlugin.register("prefix_hmac", PrefixHMACScheme())
print("Registered encrypted schemes:", EncryptedSchemePlugin.registered_names())


def run_builtin_scheme_once(scheme: str) -> dict:
    corpus = (
        CorpusBuilder(PrivacyProtocol.ENCRYPTED_SEARCH, backend_url=base_url)
        .with_encrypted_search_scheme(scheme)
        .with_chunk_size(180)
        .add_documents(docs)
        .build_local(workers=2)
    )
    cfg = PrivacyConfig(
        protocol=PrivacyProtocol.ENCRYPTED_SEARCH,
        backend=base_url,
        top_k=3,
        encrypted_search_scheme=scheme,
    )
    agent = SecureRAGAgent.from_config(cfg, corpus=corpus, llm=OllamaLLM(use_ollama=False))
    rows = agent.retriever.retrieve("q3 risk vendor concentration", round_n=0)
    return {
        "scheme": scheme,
        "mode": "backend-end-to-end",
        "top_doc": rows[0].doc_id if rows else None,
        "top_score": round(rows[0].score, 6) if rows else None,
        "context_size": len(rows),
    }


def run_custom_scheme_local_prototype() -> dict:
    # Realistic researcher prototype step: validate custom scheme behavior in-process
    # before wiring it to a standalone backend service.
    plugin = EncryptedSchemePlugin.get("prefix_hmac")
    key = plugin.generate_key()
    chunks = [
        {
            "doc_id": d.doc_id,
            "text": d.text,
            "metadata": d.metadata,
            "scheme_data": plugin.prepare_chunk(d.text, key),
        }
        for d in docs
    ]
    index = plugin.build_server_index(chunks)
    q = plugin.encrypt_query("q3 risk vendor concentration", key)
    rows = plugin.search(index, q, top_k=3)
    return {
        "scheme": "prefix_hmac",
        "mode": "in-process-prototype",
        "top_doc": rows[0]["doc_id"] if rows else None,
        "top_score": round(rows[0]["score"], 6) if rows else None,
        "context_size": len(rows),
    }


for name in ["sse", "structured"]:
    print(run_builtin_scheme_once(name))
print(run_custom_scheme_local_prototype())

Registered encrypted schemes: ['prefix_hmac', 'sse', 'structured', 'structured_encryption']
{'scheme': 'sse', 'mode': 'backend-end-to-end', 'top_doc': 'q3', 'top_score': 0.444444, 'context_size': 2}
{'scheme': 'structured', 'mode': 'backend-end-to-end', 'top_doc': 'q3', 'top_score': 0.333333, 'context_size': 2}
{'scheme': 'prefix_hmac', 'mode': 'in-process-prototype', 'top_doc': 'q3', 'top_score': 0.444444, 'context_size': 2}


## 3) Prototype a Custom Scheme and Compare

This is the typical developer workflow: define one scheme class in-notebook, register it, and run the same retrieval stack without editing framework source.

## 4) Version Guard Demonstration

Indexes marked with a legacy encrypted-search version should be rejected with a rebuild message.

In [11]:
backend = create_backend(base_url)
sse_plugin = EncryptedSchemePlugin.get("sse")
legacy_key = "0123456789abcdef0123456789abcdef"

chunks = [
    {
        "doc_id": d.doc_id,
        "text": d.text,
        "metadata": d.metadata,
        "scheme_data": sse_plugin.prepare_chunk(d.text, legacy_key),
    }
    for d in docs
]

legacy_idx = backend.build_index(
    "EncryptedSearch",
    chunks,
    encrypted_search_scheme="sse",
    encrypted_search_version="sha256-v0",
)

try:
    backend.encrypted_search(
        legacy_idx["index_id"],
        sse_plugin.encrypt_query("q3 risk", legacy_key),
        3,
    )
except BackendError as exc:
    print("Expected guard triggered:")
    print(exc)

Expected guard triggered:
Index built with crypto version 'sha256-v0' is incompatible with current 'hmac-sha256-v1'. Rebuild the corpus.


In [12]:
# Final cleanup.
if sim_proc.poll() is None:
    sim_proc.terminate()
    try:
        sim_proc.wait(timeout=2)
    except subprocess.TimeoutExpired:
        sim_proc.kill()
print("sim_server stopped")

sim_server stopped
